In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama,OllamaEmbeddings
from langchain_chroma import Chroma

In [4]:
# Load Documents
loader=PyPDFLoader("../Data/DT_Gene_Fairoz.pdf")
data=loader.load()

# Spliter
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.split_documents(data)

# Embeddings
embeddings=OllamaEmbeddings(model="nomic-embed-text")


In [6]:
vector_store=Chroma.from_documents(
    embedding=embeddings,
    documents=chunks,
    persist_directory="./vector_db_rag"
)

In [8]:
query="what is his experience?"
res=vector_store.similarity_search(query,k=2)
res

[Document(id='ca8ba05e-2161-41c0-9d45-21e3056cf544', metadata={'moddate': '2026-10-04T05:17:42+00:00', 'page_label': '1', 'keywords': '', 'creationdate': '2026-10-04T05:17:42+00:00', 'page': 0, 'total_pages': 3, 'subject': '', 'source': '../Data/DT_Gene_Fairoz.pdf', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'creator': 'LaTeX with hyperref', 'author': '', 'producer': 'pdfTeX-1.40.29', 'trapped': '/False', 'title': ''}, page_content='•Safety & Robustness:Adversarial thinking from cybersecurity background, probing models for weaknesses,\nidentifying unsafe, incorrect, or inconsistent behavior\nProfessional Experience\nAI Data T rainer — Invisible T echnologiesRemote (Contract) 2022 – Present\n•Evaluate, correct, and rank model-generated Python and technical responses against correctness, reasoning quality,\nand instruction-following criteria for large language model training pipelines.\n•Author reference solutions and stru

In [9]:
context=""
for result in res:
    context+=result.page_content

context

'•Safety & Robustness:Adversarial thinking from cybersecurity background, probing models for weaknesses,\nidentifying unsafe, incorrect, or inconsistent behavior\nProfessional Experience\nAI Data T rainer — Invisible T echnologiesRemote (Contract) 2022 – Present\n•Evaluate, correct, and rank model-generated Python and technical responses against correctness, reasoning quality,\nand instruction-following criteria for large language model training pipelines.\n•Author reference solutions and structured critiques used as supervision data, clearly explaining where and why a\nresponse succeeds or fails.\n•Apply rubric-driven review standards across tasks, maintaining a high task acceptance rate under strict quality\nreview.\n•Follow detailed project guidelines and adapt quickly to new task types, rubrics, and updated instructions.\nAI T rainer (Project-Based) — SuperAnnotate, SME Careers, AlignerrRemotewritten communication, attention to detail, and a record of maintaining high acceptance ra

In [13]:
llm=ChatOllama(model="gemma4:31b-cloud")
answer=llm.invoke(f" question:{query} context:{context}  answer from this context only  if you dont have enough information say that i dont know ")

In [14]:
answer.content

'His experience includes:\n\n*   **AI Data Trainer at Invisible Technologies (Remote Contract, 2022–Present):** He evaluates, corrects, and ranks model-generated Python and technical responses based on reasoning quality, correctness, and instruction-following. He also authors reference solutions and structured critiques for supervision data and applies rubric-driven review standards.\n*   **AI Trainer (Project-Based) at SuperAnnotate, SME Careers, and Alignerr:** He has a record of maintaining high acceptance rates under strict quality review and utilizes strong written communication and attention to detail.\n*   **Technical Skills/Competencies:** He has experience in model evaluation and ranking, data creation and annotation, quality assurance via rubric-driven reviews, Python code evaluation and debugging, and safety/robustness probing using adversarial thinking from a cybersecurity background.\n*   **Education:** He holds an M.Tech in Artificial Intelligence with a CGPA of 9.0/10.'

# **Pipeline**

In [ ]:
query=input("Ask Question :")

In [15]:
def get_context(query:str):
    res=vector_store.similarity_search(query=query)
    context=""
    for result in res:
        context+=result.page_content
    return {
        "context":context,
        "question":query
    }

In [16]:
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template('''
You are a helpful assistant and provide answers based on context only.
If anything relevant is not in context you will say that 'i have no knowledge about it'.capitalize
Context:{context}
Question:{question} 
''')

In [17]:
rag_chain=get_context | prompt | llm


In [18]:
res=rag_chain.invoke("what is his contact details and what about his acheivements?")
res.content

'His contact details are:\n* Phone: +91-7006299357\n* Email: fairozahmadsheikh@gmail.com\n\nHis achievements are:\n* Ranked among the top cybersecurity learners in India and achieved a Top 0.1% global ranking on TryHackMe.\n* Maintained a 500+ day coding streak on GitHub.'